In [1]:
from google.colab import drive
from pathlib import Path
import os

# Check if Google Drive is already mounted
if not os.path.exists('/content/drive/MyDrive'):
    print("Mounting Google Drive...")
    drive.mount('/content/drive')
else:
    print("Google Drive is already mounted.")

Mounting Google Drive...
Mounted at /content/drive


In [2]:
# ============================================================
# CELL 1: EXPERIMENT CONFIGURATION
# ============================================================

import json
import random
import unicodedata
from pathlib import Path
from collections import defaultdict
from typing import List, Tuple, Dict, Set, Optional

import pandas as pd
import matplotlib.pyplot as plt


# -----------------------------
# Reproducibility
# -----------------------------

RANDOM_SEED = 42
random.seed(RANDOM_SEED)


# -----------------------------
# Paths
# -----------------------------

# BASE_DIR = Path("/content")  # Change if your project uses another location

BASE_DIR = Path("/content/drive/MyDrive/Thesis/Structural_MorphBPE_Experiment/Structural_MorphBPE_Experiment")
MODEL_DIR = BASE_DIR / "models"
RESULT_DIR = BASE_DIR / "results"
DATA_PATH = BASE_DIR / "New_igbo_morph_scoped_corpus.json"

for d in [MODEL_DIR, RESULT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Environment ready at:", BASE_DIR)


# -----------------------------
# Experimental conditions
# -----------------------------

TARGET_VOCAB_SIZES = [2000, 4000, 8000, 12000, 16000]

TRAIN_RATIO = 0.90
EVAL_SAMPLE_SIZE = 10_000


print("Experiment configuration")
print("-" * 50)
print(f"Random seed:       {RANDOM_SEED}")
print(f"Training ratio:    {TRAIN_RATIO}")
print(f"Evaluation sample: {EVAL_SAMPLE_SIZE}")
print(f"Vocabulary sizes:  {TARGET_VOCAB_SIZES}")
print(f"Data path:         {DATA_PATH}")
print(f"Model directory:   {MODEL_DIR}")
print(f"Results directory: {RESULT_DIR}")

Environment ready at: /content/drive/MyDrive/Thesis/Structural_MorphBPE_Experiment/Structural_MorphBPE_Experiment
Experiment configuration
--------------------------------------------------
Random seed:       42
Training ratio:    0.9
Evaluation sample: 10000
Vocabulary sizes:  [2000, 4000, 8000, 12000, 16000]
Data path:         /content/drive/MyDrive/Thesis/Structural_MorphBPE_Experiment/Structural_MorphBPE_Experiment/New_igbo_morph_scoped_corpus.json
Model directory:   /content/drive/MyDrive/Thesis/Structural_MorphBPE_Experiment/Structural_MorphBPE_Experiment/models
Results directory: /content/drive/MyDrive/Thesis/Structural_MorphBPE_Experiment/Structural_MorphBPE_Experiment/results


In [3]:
# ============================================================
# CELL 2: LOAD MORPHOLOGICALLY ANNOTATED CORPUS
# ============================================================

with open(DATA_PATH, "r", encoding="utf-8") as f:
    raw_corpus_data = json.load(f)

print("Corpus loaded successfully.")
print(f"Number of word instances: {len(raw_corpus_data):,}")

print("\nExample entry:")
print(raw_corpus_data[0])

Corpus loaded successfully.
Number of word instances: 833,843

Example entry:
[['S', 'o', 'w', 'o', 'r', 'e']]


In [4]:
print("DATA_PATH:")
print(DATA_PATH)

print("\nFirst 20 entries:")
for i, entry in enumerate(raw_corpus_data[:20]):
    print(i, repr(entry))

print("\nStructural diagnostic:")
multi_morpheme = 0
single_morpheme = 0

for entry in raw_corpus_data:
    if isinstance(entry, list) and len(entry) == 1:
        single_morpheme += 1
    elif isinstance(entry, list) and len(entry) > 1:
        multi_morpheme += 1

print("Single-inner-list entries:", single_morpheme)
print("Multi-inner-list entries:", multi_morpheme)

DATA_PATH:
/content/drive/MyDrive/Thesis/Structural_MorphBPE_Experiment/Structural_MorphBPE_Experiment/New_igbo_morph_scoped_corpus.json

First 20 entries:
0 [['S', 'o', 'w', 'o', 'r', 'e']]
1 [['R', 'e', 'v', 'o', 'l', 'u', 't', 'i', 'o', 'n', ' ', ':']]
2 [['k', 'a']]
3 [['e', 'kw', 'e']]
4 [['s', 'i']]
5 [['a', 'k', 'ụ']]
6 [['m', 'a', 'k', 'a']]
7 [['n', 'g', 'a', 'gh', 'a', 'r', 'ị'], ['i', 'w', 'e']]
8 [['e']]
9 [['j', 'i']]
10 [['m', 'a', 'k', 'a']]
11 [['y', 'a']]
12 [['nw', 'ụ'], ['ch', 'i'], ['e']]
13 [['S', 'o', 'w', 'o', 'r', 'e']]
14 [['B', 'B', 'C']]
15 [['i', 'gb', 'o']]
16 [['kw', 'ụ']]
17 [['ch', 'ị'], ['m']]
18 [['i'], ['w', 'e'], ['t', 'a'], ['r', 'a']]
19 [['g', 'ị']]

Structural diagnostic:
Single-inner-list entries: 649278
Multi-inner-list entries: 184565


In [5]:
# ============================================================
# CELL 3: DEFINE ORTHOGRAPHIC UNITIZATION
# ============================================================

IGBO_DIGRAPHS = {
    "ch", "gb", "gh", "gw", "kp", "kw", "nw", "ny", "sh",
    "Ch", "Gb", "Gh", "Gw", "Kp", "Kw", "Nw", "Ny", "Sh",
    "CH", "GB", "GH", "GW", "KP", "KW", "NW", "NY", "SH"
}


def normalize_nfc(text: str) -> str:
    """
    Normalize text using Unicode NFC.
    """
    return unicodedata.normalize("NFC", text)


def unitize_morpheme(morpheme: str) -> Tuple[str, ...]:
    """
    Convert a morpheme into orthographic units.

    Igbo digraphs are treated as single units.
    All other characters are treated as individual units.
    """
    norm_m = normalize_nfc(morpheme)

    units = []
    i = 0
    n = len(norm_m)

    while i < n:
        if i + 1 < n and norm_m[i:i+2] in IGBO_DIGRAPHS:
            units.append(norm_m[i:i+2])
            i += 2
        else:
            units.append(norm_m[i])
            i += 1

    return tuple(units)


print("Orthographic unitization defined.")

example = "anyị"
print(f"Example: {example}")
print(f"Units:   {unitize_morpheme(example)}")

Orthographic unitization defined.
Example: anyị
Units:   ('a', 'ny', 'ị')


In [6]:
# ============================================================
# CELL 4: CREATE TRAINING AND EVALUATION SPLIT
# ============================================================

shuffled_data = list(raw_corpus_data)
random.shuffle(shuffled_data)

split_idx = int(TRAIN_RATIO * len(shuffled_data))

train_raw = shuffled_data[:split_idx]
eval_raw = shuffled_data[split_idx:]

# Fixed evaluation sample for computational consistency
eval_test_sample = eval_raw[:EVAL_SAMPLE_SIZE]

print("Experimental split")
print("-" * 50)
print(f"Total instances:       {len(raw_corpus_data):,}")
print(f"Training instances:    {len(train_raw):,}")
print(f"Evaluation instances:  {len(eval_raw):,}")
print(f"Evaluation sample:     {len(eval_test_sample):,}")
print(f"Training proportion:   {len(train_raw)/len(raw_corpus_data):.2%}")

Experimental split
--------------------------------------------------
Total instances:       833,843
Training instances:    750,458
Evaluation instances:  83,385
Evaluation sample:     10,000
Training proportion:   90.00%


In [7]:
# ============================================================
# CELL 5: BUILD MORPHOLOGY-AWARE AND BASELINE BPE CORPORA
# ============================================================

from collections import defaultdict
from typing import Dict, List, Tuple

# ------------------------------------------------------------
# Morphologically-aware representation
#
# Input schema:
#   [
#       ['n', 'w', 'ụ'],       # morpheme 1
#       ['c', 'h', 'i'],       # morpheme 2
#       ['e']                  # morpheme 3
#   ]
#
# Output:
#   (
#       ('nw', 'ụ'),           # morpheme 1
#       ('ch', 'i'),           # morpheme 2
#       ('e',)                 # morpheme 3
#   )
#
# The outer tuple preserves morphological boundaries.
# ------------------------------------------------------------

def clean_morpheme_chars(chars: List[str]) -> str:
    """
    Convert one annotated morpheme from character-list form
    into a normalized surface string.

    Whitespace characters are removed because they are
    orthographic separators, not morphological material.
    Other characters, including punctuation, are retained.
    """
    text = "".join(chars)

    # Remove whitespace only; retain punctuation and letters.
    text = "".join(ch for ch in text if not ch.isspace())

    return normalize_nfc(text)


def build_morph_frequency_dict(
    raw_data
) -> Dict[Tuple[Tuple[str, ...], ...], int]:
    """
    Build the morphology-aware BPE training corpus.

    Each input instance is:
        list[morpheme]
    where each morpheme is:
        list[character]

    The resulting WordTuple retains the morphological
    boundaries between morphemes.
    """
    corpus_freqs = defaultdict(int)

    for word_morphemes in raw_data:

        if not word_morphemes:
            continue

        morpheme_units = []

        for chars in word_morphemes:

            if not chars:
                continue

            morpheme = clean_morpheme_chars(chars)

            # Ignore empty morphemes created entirely from whitespace.
            if not morpheme:
                continue

            units = unitize_morpheme(morpheme)

            if units:
                morpheme_units.append(units)

        if morpheme_units:
            word_tuple = tuple(morpheme_units)
            corpus_freqs[word_tuple] += 1

    return dict(corpus_freqs)


def build_baseline_frequency_dict(
    raw_data
) -> Dict[Tuple[Tuple[str, ...], ...], int]:
    """
    Build the unsegmented baseline BPE corpus.

    The same scoped word instances are used, but morphological
    boundaries are flattened before BPE training.

    Thus:
        [['a'], ['g', 'a']]

    becomes:
        (('a', 'g', 'a'),)
    """
    corpus_freqs = defaultdict(int)

    for word_morphemes in raw_data:

        if not word_morphemes:
            continue

        all_units = []

        for chars in word_morphemes:

            if not chars:
                continue

            morpheme = clean_morpheme_chars(chars)

            if not morpheme:
                continue

            all_units.extend(unitize_morpheme(morpheme))

        if all_units:
            corpus_freqs[(tuple(all_units),)] += 1

    return dict(corpus_freqs)


# ------------------------------------------------------------
# Build both training representations from the SAME scoped data
# ------------------------------------------------------------

morph_corpus = build_morph_frequency_dict(train_raw)
baseline_corpus = build_baseline_frequency_dict(train_raw)

print("BPE training corpora constructed")
print("-" * 60)
print(f"MorphBPE types:   {len(morph_corpus):,}")
print(f"Baseline types:   {len(baseline_corpus):,}")

BPE training corpora constructed
------------------------------------------------------------
MorphBPE types:   42,754
Baseline types:   42,497


In [8]:
# ============================================================
# CELL 6: VALIDATE BPE CORPUS REPRESENTATIONS
# ============================================================

def find_whitespace_units(corpus):
    found = set()

    for word_tuple in corpus:
        for morpheme in word_tuple:
            for unit in morpheme:
                if any(ch.isspace() for ch in unit):
                    found.add(unit)

    return found


def count_morpheme_structures(corpus):
    """
    Count how many corpus types contain multiple morphemes.
    """
    single = 0
    multi = 0

    for word_tuple in corpus:
        if len(word_tuple) == 1:
            single += 1
        else:
            multi += 1

    return single, multi


morph_whitespace = find_whitespace_units(morph_corpus)
baseline_whitespace = find_whitespace_units(baseline_corpus)

morph_single, morph_multi = count_morpheme_structures(morph_corpus)

print("Corpus validation")
print("=" * 60)

print(f"MorphBPE types:              {len(morph_corpus):,}")
print(f"Baseline types:              {len(baseline_corpus):,}")

print(f"\nMorphBPE whitespace units:   {morph_whitespace}")
print(f"Baseline whitespace units:   {baseline_whitespace}")

print(f"\nMorphBPE single-morpheme:    {morph_single:,}")
print(f"MorphBPE multi-morpheme:     {morph_multi:,}")

Corpus validation
MorphBPE types:              42,754
Baseline types:              42,497

MorphBPE whitespace units:   set()
Baseline whitespace units:   set()

MorphBPE single-morpheme:    22,843
MorphBPE multi-morpheme:     19,911


In [9]:
# ============================================================
# CELL 7: INSPECT RECONSTRUCTED MORPHOLOGICAL STRUCTURE
# ============================================================

print("Sample MorphBPE representations")
print("=" * 60)

shown = 0

for word_tuple, freq in morph_corpus.items():

    if len(word_tuple) > 1:

        print(f"Frequency: {freq}")
        print("Morphemes:")
        for i, morpheme in enumerate(word_tuple):
            print(f"  {i}: {morpheme}")

        print("Surface:")
        print("".join("".join(m) for m in word_tuple))

        print("-" * 60)

        shown += 1

        if shown >= 10:
            break

Sample MorphBPE representations
Frequency: 2
Morphemes:
  0: ('gw', 'o')
  1: ('gw', 'o')
Surface:
gwogwo
------------------------------------------------------------
Frequency: 350
Morphemes:
  0: ('gb', 'u')
  1: ('r', 'u')
Surface:
gburu
------------------------------------------------------------
Frequency: 626
Morphemes:
  0: ('g', 'a', '-', 'a')
  1: ('m', 'a')
  2: ('s', 'ị')
Surface:
ga-amasị
------------------------------------------------------------
Frequency: 515
Morphemes:
  0: ('kw', 'u')
  1: ('o',)
Surface:
kwuo
------------------------------------------------------------
Frequency: 38
Morphemes:
  0: ('e',)
  1: ('l', 'e')
Surface:
ele
------------------------------------------------------------
Frequency: 9
Morphemes:
  0: ('ị',)
  1: ('h', 'a')
  2: ('z', 'i')
  3: ('gh', 'a')
  4: ('r', 'ị')
Surface:
ịhazigharị
------------------------------------------------------------
Frequency: 900
Morphemes:
  0: ('ch', 'ọ')
  1: ('r', 'ọ')
Surface:
chọrọ
----------------------

In [10]:
# ============================================================
# CELL 8: VERIFY MORPH/BASELINE SURFACE EQUIVALENCE
# ============================================================

def surface_from_word_tuple(word_tuple):
    return "".join(
        unit
        for morpheme in word_tuple
        for unit in morpheme
    )


morph_surface_counts = defaultdict(int)

for word_tuple, freq in morph_corpus.items():
    surface = surface_from_word_tuple(word_tuple)
    morph_surface_counts[surface] += freq


baseline_surface_counts = defaultdict(int)

for word_tuple, freq in baseline_corpus.items():
    surface = surface_from_word_tuple(word_tuple)
    baseline_surface_counts[surface] += freq


print("Surface-equivalence validation")
print("=" * 60)

print(
    "MorphBPE total instances:",
    f"{sum(morph_surface_counts.values()):,}"
)

print(
    "Baseline total instances:",
    f"{sum(baseline_surface_counts.values()):,}"
)

print(
    "MorphBPE unique surfaces:",
    f"{len(morph_surface_counts):,}"
)

print(
    "Baseline unique surfaces:",
    f"{len(baseline_surface_counts):,}"
)

print(
    "Identical surface inventories:",
    morph_surface_counts.keys() == baseline_surface_counts.keys()
)

print(
    "Identical surface frequencies:",
    morph_surface_counts == baseline_surface_counts
)

Surface-equivalence validation
MorphBPE total instances: 750,457
Baseline total instances: 750,457
MorphBPE unique surfaces: 42,496
Baseline unique surfaces: 42,496
Identical surface inventories: True
Identical surface frequencies: True


In [11]:
# ============================================================
# CELL 9: IDENTIFY SURFACE-EQUIVALENCE MISMATCHES
# ============================================================

morph_only = set(morph_surface_counts) - set(baseline_surface_counts)
baseline_only = set(baseline_surface_counts) - set(morph_surface_counts)

frequency_mismatches = {
    surface: (
        morph_surface_counts[surface],
        baseline_surface_counts[surface]
    )
    for surface in morph_surface_counts.keys() & baseline_surface_counts.keys()
    if morph_surface_counts[surface] != baseline_surface_counts[surface]
}

print("Surface-equivalence mismatch analysis")
print("=" * 60)

print(f"Surfaces only in MorphBPE:    {len(morph_only):,}")
print(f"Surfaces only in Baseline:    {len(baseline_only):,}")
print(f"Frequency mismatches:         {len(frequency_mismatches):,}")

print("\nExamples: MorphBPE only")
for surface in list(morph_only)[:10]:
    print(repr(surface), morph_surface_counts[surface])

print("\nExamples: Baseline only")
for surface in list(baseline_only)[:10]:
    print(repr(surface), baseline_surface_counts[surface])

print("\nExamples: frequency mismatches")
for surface, counts in list(frequency_mismatches.items())[:10]:
    print(
        repr(surface),
        "MorphBPE =", counts[0],
        "Baseline =", counts[1]
    )

Surface-equivalence mismatch analysis
Surfaces only in MorphBPE:    0
Surfaces only in Baseline:    0
Frequency mismatches:         0

Examples: MorphBPE only

Examples: Baseline only

Examples: frequency mismatches


In [12]:
# ============================================================
# CELL 10: DIRECT INSTANCE-LEVEL SURFACE VALIDATION
# ============================================================

def surface_from_raw_instance(word_morphemes):
    parts = []

    for chars in word_morphemes:
        if not chars:
            continue

        text = "".join(ch for ch in chars if not ch.isspace())
        text = normalize_nfc(text)

        if text:
            parts.append(text)

    return "".join(parts)


def morph_surface_from_instance(word_morphemes):
    morpheme_units = []

    for chars in word_morphemes:
        if not chars:
            continue

        morpheme = clean_morpheme_chars(chars)

        if not morpheme:
            continue

        units = unitize_morpheme(morpheme)

        if units:
            morpheme_units.append(units)

    word_tuple = tuple(morpheme_units)

    return surface_from_word_tuple(word_tuple)


def baseline_surface_from_instance(word_morphemes):
    all_units = []

    for chars in word_morphemes:
        if not chars:
            continue

        morpheme = clean_morpheme_chars(chars)

        if not morpheme:
            continue

        all_units.extend(unitize_morpheme(morpheme))

    word_tuple = (tuple(all_units),)

    return surface_from_word_tuple(word_tuple)


# ------------------------------------------------------------
# Compare every raw training instance
# ------------------------------------------------------------

raw_vs_morph_mismatches = []
raw_vs_baseline_mismatches = []
morph_vs_baseline_mismatches = []

for idx, word_morphemes in enumerate(train_raw):

    raw_surface = surface_from_raw_instance(word_morphemes)
    morph_surface = morph_surface_from_instance(word_morphemes)
    baseline_surface = baseline_surface_from_instance(word_morphemes)

    if raw_surface != morph_surface:
        raw_vs_morph_mismatches.append(
            (idx, word_morphemes, raw_surface, morph_surface)
        )

    if raw_surface != baseline_surface:
        raw_vs_baseline_mismatches.append(
            (idx, word_morphemes, raw_surface, baseline_surface)
        )

    if morph_surface != baseline_surface:
        morph_vs_baseline_mismatches.append(
            (idx, word_morphemes, morph_surface, baseline_surface)
        )


print("Direct instance-level validation")
print("=" * 60)

print(
    "Raw → MorphBPE mismatches:",
    len(raw_vs_morph_mismatches)
)

print(
    "Raw → Baseline mismatches:",
    len(raw_vs_baseline_mismatches)
)

print(
    "MorphBPE → Baseline mismatches:",
    len(morph_vs_baseline_mismatches)
)

Direct instance-level validation
Raw → MorphBPE mismatches: 0
Raw → Baseline mismatches: 0
MorphBPE → Baseline mismatches: 0


In [13]:
# ============================================================
# CELL 11: SHOW FIRST DIRECT MISMATCH
# ============================================================

if morph_vs_baseline_mismatches:

    idx, raw_entry, morph_surface, baseline_surface = (
        morph_vs_baseline_mismatches[0]
    )

    print("First MorphBPE/Baseline mismatch")
    print("=" * 60)

    print("Index:", idx)
    print("\nRaw entry:")
    print(raw_entry)

    print("\nRaw surface:")
    print(repr(surface_from_raw_instance(raw_entry)))

    print("\nMorphBPE surface:")
    print(repr(morph_surface))

    print("\nBaseline surface:")
    print(repr(baseline_surface))

else:
    print("No direct MorphBPE/Baseline mismatches found.")

No direct MorphBPE/Baseline mismatches found.


In [14]:
# ============================================================
# CELL 12: FINAL SURFACE EQUIVALENCE CHECK
# ============================================================

raw_surface_counts = defaultdict(int)
morph_surface_counts_final = defaultdict(int)
baseline_surface_counts_final = defaultdict(int)

for word_morphemes in train_raw:

    # Authoritative surface
    raw_surface = surface_from_raw_instance(word_morphemes)
    raw_surface_counts[raw_surface] += 1

    # MorphBPE representation
    morph_surface = morph_surface_from_instance(word_morphemes)
    morph_surface_counts_final[morph_surface] += 1

    # Baseline representation
    baseline_surface = baseline_surface_from_instance(word_morphemes)
    baseline_surface_counts_final[baseline_surface] += 1


print("FINAL SURFACE-EQUIVALENCE VALIDATION")
print("=" * 60)

print(f"Raw instances:       {sum(raw_surface_counts.values()):,}")
print(f"MorphBPE instances:  {sum(morph_surface_counts_final.values()):,}")
print(f"Baseline instances:  {sum(baseline_surface_counts_final.values()):,}")

print()
print(f"Raw unique surfaces:       {len(raw_surface_counts):,}")
print(f"MorphBPE unique surfaces:  {len(morph_surface_counts_final):,}")
print(f"Baseline unique surfaces:  {len(baseline_surface_counts_final):,}")

print()
print(
    "Raw == MorphBPE:",
    raw_surface_counts == morph_surface_counts_final
)

print(
    "Raw == Baseline:",
    raw_surface_counts == baseline_surface_counts_final
)

print(
    "MorphBPE == Baseline:",
    morph_surface_counts_final == baseline_surface_counts_final
)

FINAL SURFACE-EQUIVALENCE VALIDATION
Raw instances:       750,458
MorphBPE instances:  750,458
Baseline instances:  750,458

Raw unique surfaces:       42,497
MorphBPE unique surfaces:  42,497
Baseline unique surfaces:  42,497

Raw == MorphBPE: True
Raw == Baseline: True
MorphBPE == Baseline: True


In [20]:
# ============================================================
# CELL 13: MORPHBPE TRAINER
# ============================================================

from collections import defaultdict
from typing import Dict, List, Tuple, Set, Optional

WordTuple = Tuple[Tuple[str, ...], ...]

class MorphBPETrainer:

    def __init__(self, special_tokens: Optional[List[str]] = None):

        self.special_tokens = special_tokens or [
            "<unk>",
            "<s>",
            "</s>",
            "<pad>",
            "<mask morph>"
        ]

        self.merges = []
        self.vocab = {}

    def _extract_word_pairs(
        self,
        word_tuple: WordTuple
    ) -> List[Tuple[str, str]]:

        pairs = []

        # IMPORTANT:
        # Pairs are extracted separately inside each morpheme.
        # Therefore, cross-morpheme pairs are never counted.

        for morpheme in word_tuple:

            for i in range(len(morpheme) - 1):

                pairs.append(
                    (morpheme[i], morpheme[i + 1])
                )

        return pairs

    def _apply_merge_to_morpheme(
        self,
        morpheme: Tuple[str, ...],
        pair: Tuple[str, str]
    ) -> Tuple[str, ...]:

        first, second = pair
        merged = first + second

        new_morpheme = []

        i = 0
        n = len(morpheme)

        while i < n:

            if (
                i < n - 1
                and morpheme[i] == first
                and morpheme[i + 1] == second
            ):
                new_morpheme.append(merged)
                i += 2

            else:
                new_morpheme.append(morpheme[i])
                i += 1

        return tuple(new_morpheme)

    def train(
        self,
        corpus: Dict[WordTuple, int],
        target_vocab_size: int
    ):

        self.merges = []

        initial_units = {
            unit
            for word in corpus
            for morpheme in word
            for unit in morpheme
        }

        current_vocab = (
            list(self.special_tokens)
            + sorted(list(initial_units))
        )

        self.vocab = {
            token: idx
            for idx, token in enumerate(current_vocab)
        }

        working_corpus = dict(corpus)

        pair_counts = defaultdict(int)
        where_pair = defaultdict(set)

        # Initial pair statistics
        for word_tuple, freq in working_corpus.items():

            for pair in self._extract_word_pairs(word_tuple):

                pair_counts[pair] += freq
                where_pair[pair].add(word_tuple)

        # BPE learning loop
        while len(self.vocab) < target_vocab_size:

            if not pair_counts:
                break

            best_pair = max(
    pair_counts,
    key=lambda p: (pair_counts[p], p)
)

            if pair_counts[best_pair] <= 1:
                break

            self.merges.append(best_pair)

            merged_token = best_pair[0] + best_pair[1]

            if merged_token not in self.vocab:
                self.vocab[merged_token] = len(self.vocab)

            words_to_update = list(
                where_pair[best_pair]
            )

            del where_pair[best_pair]
            del pair_counts[best_pair]

            for old_word in words_to_update:

                if old_word not in working_corpus:
                    continue

                freq = working_corpus.pop(old_word)

                # Remove old pair contributions
                for pair in self._extract_word_pairs(old_word):

                    if pair != best_pair:

                        pair_counts[pair] -= freq

                        if pair_counts[pair] <= 0:
                            pair_counts.pop(pair, None)

                        where_pair[pair].discard(old_word)

                # Apply merge within each morpheme
                new_word = tuple(
                    self._apply_merge_to_morpheme(
                        morpheme,
                        best_pair
                    )
                    for morpheme in old_word
                )

                working_corpus[new_word] = (
                    working_corpus.get(new_word, 0)
                    + freq
                )

                # Add new pair contributions
                for pair in self._extract_word_pairs(new_word):

                    pair_counts[pair] += freq
                    where_pair[pair].add(new_word)

    def export_vocab(self, filepath: str):

        with open(filepath, "w", encoding="utf-8") as f:
            json.dump(
                self.vocab,
                f,
                ensure_ascii=False,
                indent=2
            )

    def export_merges(self, filepath: str):

        with open(filepath, "w", encoding="utf-8") as f:

            f.write("#version: 0.2\n")

            for pair in self.merges:

                f.write(
                    f"{pair[0]} {pair[1]}\n"
                )


print("MorphBPE trainer defined.")

MorphBPE trainer defined.


In [21]:
# ============================================================
# CELL 14: BASELINE BPE TRAINER
# ============================================================

class BaselineBPETrainer(MorphBPETrainer):

    def _extract_word_pairs(
        self,
        word_tuple: WordTuple
    ) -> List[Tuple[str, str]]:

        pairs = []

        # The baseline receives the whole surface word
        # as a single sequence, so pairs may occur across
        # original morphological boundaries.

        for word_sequence in word_tuple:

            for i in range(len(word_sequence) - 1):

                pairs.append(
                    (
                        word_sequence[i],
                        word_sequence[i + 1]
                    )
                )

        return pairs


print("Baseline BPE trainer defined.")

Baseline BPE trainer defined.


In [18]:
# ============================================================
# CELL 15: EXPERIMENTAL VOCABULARY SIZES
# ============================================================

TARGET_VOCAB_SIZES = [
    2_000,
    4_000,
    8_000,
    12_000,
    16_000
]

print("Target vocabulary sizes:")
for size in TARGET_VOCAB_SIZES:
    print(f"  {size:,}")

Target vocabulary sizes:
  2,000
  4,000
  8,000
  12,000
  16,000


In [22]:
# ============================================================
# CELL 16: TRAIN MORPHBPE MODELS
# ============================================================

morph_models = {}

print("Training MorphBPE models")
print("=" * 60)

for target_vocab_size in TARGET_VOCAB_SIZES:

    print(
        f"\nTraining MorphBPE "
        f"(target vocabulary = {target_vocab_size:,})"
    )

    trainer = MorphBPETrainer()

    trainer.train(
        morph_corpus,
        target_vocab_size=target_vocab_size
    )

    vocab_path = (
        MODEL_DIR
        / f"m_{target_vocab_size}_vocab.json"
    )

    merges_path = (
        MODEL_DIR
        / f"m_{target_vocab_size}_merges.txt"
    )

    trainer.export_vocab(vocab_path)
    trainer.export_merges(merges_path)

    morph_models[target_vocab_size] = {
        "vocab_size": len(trainer.vocab),
        "num_merges": len(trainer.merges),
        "vocab_path": vocab_path,
        "merges_path": merges_path
    }

    print(
        f"Actual vocabulary: {len(trainer.vocab):,}"
    )

    print(
        f"Learned merges:    {len(trainer.merges):,}"
    )

print("\nMorphBPE training complete.")

Training MorphBPE models

Training MorphBPE (target vocabulary = 2,000)
Actual vocabulary: 2,000
Learned merges:    1,818

Training MorphBPE (target vocabulary = 4,000)
Actual vocabulary: 4,000
Learned merges:    3,818

Training MorphBPE (target vocabulary = 8,000)
Actual vocabulary: 8,000
Learned merges:    7,818

Training MorphBPE (target vocabulary = 12,000)
Actual vocabulary: 12,000
Learned merges:    11,818

Training MorphBPE (target vocabulary = 16,000)
Actual vocabulary: 16,000
Learned merges:    15,818

MorphBPE training complete.


In [23]:
# ============================================================
# CELL 17: TRAIN BASELINE BPE MODELS
# ============================================================

baseline_models = {}

print("Training Baseline BPE models")
print("=" * 60)

for target_vocab_size in TARGET_VOCAB_SIZES:

    print(
        f"\nTraining Baseline BPE "
        f"(target vocabulary = {target_vocab_size:,})"
    )

    trainer = BaselineBPETrainer()

    trainer.train(
        baseline_corpus,
        target_vocab_size=target_vocab_size
    )

    vocab_path = (
        MODEL_DIR
        / f"b_{target_vocab_size}_vocab.json"
    )

    merges_path = (
        MODEL_DIR
        / f"b_{target_vocab_size}_merges.txt"
    )

    trainer.export_vocab(vocab_path)
    trainer.export_merges(merges_path)

    baseline_models[target_vocab_size] = {
        "vocab_size": len(trainer.vocab),
        "num_merges": len(trainer.merges),
        "vocab_path": vocab_path,
        "merges_path": merges_path
    }

    print(
        f"Actual vocabulary: {len(trainer.vocab):,}"
    )

    print(
        f"Learned merges:    {len(trainer.merges):,}"
    )

print("\nBaseline BPE training complete.")

Training Baseline BPE models

Training Baseline BPE (target vocabulary = 2,000)
Actual vocabulary: 2,000
Learned merges:    1,818

Training Baseline BPE (target vocabulary = 4,000)
Actual vocabulary: 4,000
Learned merges:    3,818

Training Baseline BPE (target vocabulary = 8,000)
Actual vocabulary: 8,000
Learned merges:    7,818

Training Baseline BPE (target vocabulary = 12,000)
Actual vocabulary: 12,000
Learned merges:    11,818

Training Baseline BPE (target vocabulary = 16,000)
Actual vocabulary: 16,000
Learned merges:    15,818

Baseline BPE training complete.
